## Transformer Sequence Generation

You've reached the final lesson on Transformer inference—how stellar is that? Ready to see your model generate sequences like a true space explorer? 🚀

---

### Introduction

Welcome back to Bringing Transformers to Life: Training & Inference! This is an extraordinary milestone as you reach the final lesson of this course. Throughout our journey together, you've assembled a complete Transformer architecture, built robust data preparation pipelines, and implemented sophisticated training procedures with teacher forcing and learning rate scheduling. You should be incredibly proud of the deep understanding you've developed of these powerful models.

Today, we shift our focus from training to inference: the art of generating sequences with your trained Transformer model. This is where the magic truly happens, as we watch our model generate coherent text one token at a time. We'll explore two fundamental inference strategies: greedy decoding for fast generation and beam search for higher-quality output. You'll learn to implement both approaches, understand their trade-offs, and see how they perform on practical examples. By the end of this lesson, you'll have a complete inference pipeline that can generate sequences from any trained Transformer model.

---

### From Training to Inference: A Different Challenge

Inference presents fundamentally different challenges compared to training. During training, we used teacher forcing, where the model always sees the correct previous tokens. During inference, however, the model must generate sequences autoregressively, using its own predictions as input for subsequent tokens. This creates a sequential dependency in which each prediction influences all future predictions.

The inference process follows this mathematical formulation: given a source sequence $x$ and previously generated tokens $y_1, y_2, ..., y_{t-1}$, we compute:

$$P(y_t \mid y_{<t}, x) = \text{softmax}(f_\theta(y_{<t}, x))_t$$

where $f_\theta$ represents our trained Transformer model. We start with a special `<SOS>` token, generate the most likely next token, append it to the sequence, and repeat until we encounter an `<EOS>` token or reach a maximum length. This autoregressive nature means that early mistakes can propagate through the entire sequence, making the choice of decoding strategy crucial for output quality.

---

### Greedy Decoding: The Simplest Strategy

Greedy decoding represents the most straightforward approach to sequence generation: at each step, we simply select the token with the highest probability. Mathematically, this means selecting:

$$y_t = \arg\max_{w \in V} P(w \mid y_{<t}, x)$$

where $V$ is our vocabulary. While this strategy is fast and deterministic, it can lead to suboptimal sequences because it never reconsiders previous choices.

Let's implement the greedy decoding method:

```python
def greedy_decode(self, src_sentence, max_len=50):
    """Greedy decoding - select most likely token at each step"""
    # Encode source sentence with consistent max_len
    src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
    src_mask = self.model.create_padding_mask(src_tokens)

    # Initialize decoder input with SOS token
    tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)

    for _ in range(max_len):
        # Create both causal and padding masks for target
        tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
        tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
        tgt_mask = tgt_causal_mask & tgt_padding_mask

        # Forward pass with no_grad for better performance
        with torch.no_grad():
            output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)

        # Get next token
        next_token_logits = output[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)

        # Stop if EOS token generated
        if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
            break

        # Append to sequence
        tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)

    return self.tgt_vocab.decode(tgt_tokens[0].tolist())
```

This implementation demonstrates the core autoregressive loop through the following key steps:

- **Source preparation:** Encode the source sentence with consistent `max_len` and create appropriate padding masks.
- **Target initialization:** Start with just the `<SOS>` token in the target sequence.
- **Iterative generation loop:** For each step up to `max_len`:
    - Create both causal and padding masks for the current target sequence;
    - Perform a forward pass through the model using `torch.no_grad()` for better performance;
    - Extract the logits for the next token position and select the highest probability token with `torch.argmax`;
    - Check if the generated token is `<EOS>` and terminate if sequence is complete;
    - Append the new token to the growing target sequence.

The `torch.no_grad()` context improves performance by disabling gradient computation during inference, which is unnecessary and computationally expensive since we're not training the model.

---

### Beam Search: Exploring Multiple Paths

Beam search offers a more sophisticated approach by maintaining multiple candidate sequences (called the "beam") and exploring several possibilities simultaneously. Instead of committing to a single choice at each step, beam search keeps track of the top-k most promising sequences and expands each one. The score for each sequence is the cumulative log probability:

$$\text{score}(y_1, ..., y_t) = \sum_{i=1}^{t} \log P(y_i \mid y_{<i}, x)$$

Here's the beam search implementation:

```python
def beam_search(self, src_sentence, beam_width=3, max_len=50):
    """Beam search decoding - maintain multiple hypotheses"""
    # Encode source with consistent max_len
    src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
    src_mask = self.model.create_padding_mask(src_tokens)

    # Initialize beam with SOS token
    beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]

    for step in range(max_len):
        candidates = []

        for beam in beams:
            if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                candidates.append(beam)
                continue

            # Get current sequence
            tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask

            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)

            # Get probabilities for next token
            logits = output[:, -1, :]
            probs = F.log_softmax(logits, dim=-1)

            # Get top-k candidates
            top_probs, top_indices = torch.topk(probs, beam_width)

            for prob, idx in zip(top_probs[0], top_indices[0]):
                new_tokens = beam['tokens'] + [idx.item()]
                new_score = beam['score'] + prob.item()
                candidates.append({'tokens': new_tokens, 'score': new_score})

        # Select top beam_width candidates
        candidates.sort(key=lambda x: x['score'], reverse=True)
        beams = candidates[:beam_width]

        # Check if all beams ended
        if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
            break

    # Return best sequence
    best_beam = max(beams, key=lambda x: x['score'])
    return self.tgt_vocab.decode(best_beam['tokens'])
```

Beam search maintains a list of candidate sequences with their accumulated log probability scores. This implementation demonstrates the core beam search algorithm through the following key steps:

- **Beam initialization:** Start with a single beam containing just the `<SOS>` token with score `0.0`.
- **Iterative expansion loop:** For each step up to `max_len`:
    - **Beam processing:** Iterate through current beams, skipping those that have already terminated with `<EOS>`;
    - **Forward pass:** For each active beam, prepare target tokens and masks, then perform a forward pass using `torch.no_grad()`;
    - **Probability calculation:** Extract logits and apply `log_softmax` to get log probabilities, avoiding numerical underflow;
    - **Candidate generation:** Use `torch.topk` to get the top-k most probable next tokens for each beam;
    - **Beam expansion:** Create new candidate sequences by extending each active beam with its top-k tokens;
    - **Beam selection:** Sort all candidates by accumulated score and select the top `beam_width` sequences.
- **Termination:** Continue until all beams end with `<EOS>` tokens or maximum length is reached.
- **Result selection:** Return the sequence with the highest accumulated log probability score.

The key insight is using `log_softmax` to get log probabilities, which we accumulate by addition rather than multiplication, avoiding numerical underflow issues. After expanding all beams, the algorithm selects the top `beam_width` candidates based on their scores and continues until all beams terminate with `<EOS>` tokens.

---

### Setting Up the Inference Pipeline

The `TransformerInference` class encapsulates our inference functionality and provides a clean interface for generating sequences:

```python
class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
```

This initialization is straightforward but crucial. We store references to the trained model and both vocabularies, then call `model.eval()` to disable dropout and batch normalization training behaviors. This ensures consistent inference behavior and prevents the randomness that would occur during training mode, which is essential for reproducible results.

Let's examine the complete pipeline that trains a model and tests both inference strategies:

```python
def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")

    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)

    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)

    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )

    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
```

This pipeline demonstrates the complete workflow from data preparation through training to inference testing. We create synthetic data using our word reversal task, build vocabularies, train a compact Transformer model for three epochs, and prepare it for inference evaluation. The model architecture is intentionally small to enable quick training while still demonstrating meaningful learning behavior.

---

### Testing and Analyzing Results

The inference testing reveals fascinating insights about the different strategies:

```python
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)

    test_sentences = ["hello world", "good morning", "thank you very much"]

    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")

        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time

        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time

        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
```

When we run this complete pipeline, we observe the following results:

```text
Testing Transformer Inference...
Quick training for 2 epochs...
Step 10, Loss: 2.1828, LR: 0.011000
Step 20, Loss: 1.6475, LR: 0.021000
Epoch 1, Loss: 1.5726
Step 35, Loss: 1.3399, LR: 0.020833
Step 45, Loss: 1.3126, LR: 0.018430
Epoch 2, Loss: 1.2247
Step 60, Loss: 1.2136, LR: 0.016005
Step 70, Loss: 1.2314, LR: 0.014835
Epoch 3, Loss: 1.2724

Inference Results:
Source: 'hello world'
Expected: 'world hello'
Greedy: 'world hello' (time: 0.501s)
Beam: 'world hello' (time: 3.298s)
--------------------------------------------------
Source: 'good morning'
Expected: 'morning good'
Greedy: 'morning good' (time: 0.301s)
Beam: 'morning good' (time: 0.500s)
--------------------------------------------------
Source: 'thank you very much'
Expected: 'much very you thank'
Greedy: 'you to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to to' (time: 3.199s)
Beam: 'much very you thank' (time: 3.300s)
--------------------------------------------------
```

The output reveals crucial insights about the different decoding strategies. For simple inputs like "hello world" and "good morning," both strategies produce perfect results, but greedy decoding is significantly faster (0.3 - 0.5 seconds vs. 0.5 - 3.3 seconds). However, the third example, "thank you very much," shows where greedy decoding fails catastrophically: it gets stuck in a repetitive loop, generating "you to to to..." until reaching the maximum length limit.

This repetitive behavior occurs because greedy decoding can become trapped in local probability maxima:

1. The model generates "you";
2. The sequence context makes "to" the most probable next token;
3. With "you to" in the context, the model again predicts "to" as most likely, creating a feedback loop where the model reinforces its own poor predictions.

This demonstrates why greedy decoding's myopic approach—never reconsidering previous choices—can lead to degraded output quality for complex sequences.

---

### Conclusion and Next Steps

Congratulations on completing the final lesson of Bringing Transformers to Life: Training & Inference! You have successfully implemented a complete inference pipeline that showcases the practical application of trained Transformer models. Through greedy decoding and beam search, you've learned to balance the trade-offs between speed and quality in sequence generation, understanding when each strategy is most appropriate. Your journey from building Transformers from scratch to implementing sophisticated inference strategies represents a remarkable achievement in mastering these powerful models.

You should be incredibly proud of reaching this milestone. You've built a Transformer from the ground up, created robust data pipelines, implemented sophisticated training procedures, and now mastered the art of inference. This comprehensive understanding positions you perfectly for the next course in our learning path: Harnessing Transformers with Hugging Face, where we'll dive into the modern Hugging Face ecosystem and learn to leverage state-of-the-art pre-trained models for real-world applications. The upcoming practice exercises will solidify your understanding of inference strategies before you embark on this exciting next chapter.


## Creating Your First Text Generator

Now that you understand the theory behind greedy decoding, it's time to implement the core autoregressive loop that makes it work. You'll complete the missing parts of the `greedy_decode` method in the provided code.

---

### Your Task

The method already handles the model setup and main loop structure, but you need to implement three key steps:

- Extract the logits for the last token from the model's output.
- Use `argmax` to select the token with the highest probability.
- Append this new token to the target sequence for the next iteration.

This is the foundation of autoregressive generation, so mastering this will prepare you for more advanced decoding strategies!

---

### Starter Code

```python
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # TODO: Get the logits for the last token in the sequence
            
            # TODO: Select the token with the highest probability (greedy choice)
            
            # Stop if EOS token generated
            if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
                break
            
            # TODO: Append the new token to the target sequence for the next iteration
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                
                # Get top-k candidates
                top_probs, top_indices = torch.topk(probs, beam_width)
                
                for prob, idx in zip(top_probs[0], top_indices[0]):
                    new_tokens = beam['tokens'] + [idx.item()]
                    new_score = beam['score'] + prob.item()
                    candidates.append({'tokens': new_tokens, 'score': new_score})
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])


## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()
```


---

### Solution

The loop in `greedy_decode` was missing the three steps that turn the model output into the next token and grow the target sequence:

```python
# Get the logits for the last token in the sequence
next_token_logits = output[:, -1, :]

# Select the token with the highest probability (greedy choice)
next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)

# Append the new token to the target sequence for the next iteration
tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
```

- `output[:, -1, :]` keeps only the last position of the decoder output. Its shape goes from `(batch, tgt_len, vocab_size)` to `(batch, vocab_size)`, and the last position is the prediction for the next token.
- `torch.argmax(..., dim=-1)` picks the index of the highest logit in the vocabulary, which is the greedy choice. `.unsqueeze(1)` reshapes the result from `(batch,)` to `(batch, 1)` so it can be concatenated to the sequence.
- `torch.cat([tgt_tokens, next_token], dim=1)` appends the new token, so the next iteration feeds the longer sequence back into the decoder. Without it, the model sees the same input at every step and the output never grows.
- `next_token` must be defined before the `<EOS>` check. Without the `argmax` line, `next_token.item()` raises a `NameError`.

When the script runs, it trains the small model for 2 epochs and prints the greedy and beam search results for the three test sentences, together with the time each one took.

The notebook version needs the helper modules (`transformer.model`, `data_utils`, `train`) to be available, as in the CodeSignal project.


In [ ]:
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # Get the logits for the last token in the sequence
            next_token_logits = output[:, -1, :]
            
            # Select the token with the highest probability (greedy choice)
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # Stop if EOS token generated
            if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
                break
            
            # Append the new token to the target sequence for the next iteration
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                
                # Get top-k candidates
                top_probs, top_indices = torch.topk(probs, beam_width)
                
                for prob, idx in zip(top_probs[0], top_indices[0]):
                    new_tokens = beam['tokens'] + [idx.item()]
                    new_score = beam['score'] + prob.item()
                    candidates.append({'tokens': new_tokens, 'score': new_score})
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])


## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()


## Perfect Timing for Text Generation

Now that you've implemented the core greedy decoding loop, there's still an issue with how the generation process ends. Currently, the decoding always runs for the full maximum length, which means it might keep generating tokens even after the sequence should naturally end.

In real language generation, we want the model to stop producing tokens when it generates an End-of-Sequence (`EOS`) token. This prevents the output from containing unnecessary padding tokens or repetitive text, which makes the result look unnatural.

---

### Your Task

Your job is to add a termination condition in the `greedy_decode` method. Look for where the next token is selected, and add logic to check if this token signals the end of the sequence. If it does, the generation loop should stop immediately.

This small but crucial addition will make your generated sequences much cleaner and more realistic!

---

### Starter Code

```python
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # Get next token
            next_token_logits = output[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # TODO: Add termination condition - check if next_token is EOS and break if so
            
            # Append to sequence
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                
                # Get top-k candidates
                top_probs, top_indices = torch.topk(probs, beam_width)
                
                for prob, idx in zip(top_probs[0], top_indices[0]):
                    new_tokens = beam['tokens'] + [idx.item()]
                    new_score = beam['score'] + prob.item()
                    candidates.append({'tokens': new_tokens, 'score': new_score})
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])

## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()
```


---

### Solution

The loop in `greedy_decode` never checked whether the model had produced `<EOS>`, so it always ran for the full `max_len` steps. The missing termination condition goes right after the next token is selected:

```python
# Stop if EOS token generated
if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
    break
```

- `next_token` has shape `(1, 1)`, so `.item()` turns it into a plain Python integer that can be compared with the `<EOS>` index.
- `self.tgt_vocab.token2idx['<EOS>']` looks up the integer id of the end-of-sequence token in the target vocabulary.
- `break` leaves the `for` loop immediately, so `<EOS>` is never appended to `tgt_tokens` and no further tokens are generated.
- The check must come before the `torch.cat` line. If it came after, `<EOS>` would be appended to the output before the loop stopped.
- Without the check, generation continues until `max_len` is reached, so the decoded output can contain repeated or meaningless tokens after the sentence should have ended.

When the script runs, it trains the small model for 2 epochs and prints the greedy and beam search results for the three test sentences, together with the time each one took. Greedy decoding now stops as soon as the model emits `<EOS>`.

The notebook version needs the helper modules (`transformer.model`, `data_utils`, `train`) to be available, as in the CodeSignal project.


In [ ]:
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # Get next token
            next_token_logits = output[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # Stop if EOS token generated
            if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
                break
            
            # Append to sequence
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                probs = F.log_softmax(logits, dim=-1)
                
                # Get top-k candidates
                top_probs, top_indices = torch.topk(probs, beam_width)
                
                for prob, idx in zip(top_probs[0], top_indices[0]):
                    new_tokens = beam['tokens'] + [idx.item()]
                    new_score = beam['score'] + prob.item()
                    candidates.append({'tokens': new_tokens, 'score': new_score})
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])

## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()


## Implementing Core Beam Search Logic

Now that you've mastered greedy decoding, it's time to implement the more advanced beam search algorithm, which can generate higher-quality sequences by keeping track of multiple possibilities at once.

Unlike greedy decoding, which picks only the single best token at each step, beam search maintains several promising sequences (called "beams") and explores the most likely paths forward. The core challenge is expanding each beam by considering the top candidates for the next token.

---

### Your Task

Your job is to complete the beam expansion logic in the provided code:

- Calculate log probabilities from the model's output using `F.log_softmax`.
- Find the top `beam_width` most promising next tokens with `torch.topk`.
- Create new candidate beams by extending each current beam with these top tokens and their scores.

Each new candidate should be a dictionary containing the extended token sequence and the updated cumulative score. Once you get this working, you'll see how beam search can produce much more coherent and diverse outputs than simple greedy decoding!

---

### Starter Code

```python
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # Get next token
            next_token_logits = output[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # Stop if EOS token generated
            if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
                break
            
            # Append to sequence
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                
                # TODO: Calculate log probabilities for the next token
                
                # TODO: Get the top 'beam_width' candidates (probabilities and indices)
                
                # TODO: Create new candidate beams by extending the current beam
                # with each of the top indices and their corresponding scores.
                # Append each new candidate as a dictionary to the 'candidates' list.
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])

## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()
```


---

### Solution

The beam expansion step inside `beam_search` was missing: the code computed `logits` but never turned them into new candidate beams. The three missing pieces are:

```python
# Calculate log probabilities for the next token
probs = F.log_softmax(logits, dim=-1)

# Get the top 'beam_width' candidates (probabilities and indices)
top_probs, top_indices = torch.topk(probs, beam_width)

# Create new candidate beams by extending the current beam
for prob, idx in zip(top_probs[0], top_indices[0]):
    new_tokens = beam['tokens'] + [idx.item()]
    new_score = beam['score'] + prob.item()
    candidates.append({'tokens': new_tokens, 'score': new_score})
```

- `F.log_softmax(logits, dim=-1)` converts the logits of shape `(1, vocab_size)` into log probabilities over the vocabulary. Log probabilities can be added together, which avoids the numerical underflow that multiplying many small probabilities would cause.
- `torch.topk(probs, beam_width)` returns the `beam_width` highest log probabilities and their token indices. Both have shape `(1, beam_width)`, which is why the loop reads `top_probs[0]` and `top_indices[0]`.
- `beam['tokens'] + [idx.item()]` builds a new list instead of modifying the current beam, so the same beam can be extended with several different tokens.
- `beam['score'] + prob.item()` adds the log probability of the new token to the cumulative score of the beam.
- Each candidate is appended to `candidates` as a dictionary with `'tokens'` and `'score'`. Without these lines `candidates` stays empty for every active beam, so `candidates[:beam_width]` is empty and `max(beams, ...)` raises a `ValueError` on an empty sequence.

When the script runs, it trains the small model for 2 epochs and prints the greedy and beam search results for the three test sentences, together with the time each one took.

The notebook version needs the helper modules (`transformer.model`, `data_utils`, `train`) to be available, as in the CodeSignal project.


In [ ]:
## inference.py

import torch
import torch.nn.functional as F

class TransformerInference:
    def __init__(self, model, src_vocab, tgt_vocab):
        self.model = model
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.model.eval()
    
    def greedy_decode(self, src_sentence, max_len=50):
        """Greedy decoding - select most likely token at each step"""
        # Encode source sentence with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize decoder input with SOS token
        tgt_tokens = torch.tensor([[self.tgt_vocab.token2idx['<SOS>']]], dtype=torch.long)
        
        for _ in range(max_len):
            # Create both causal and padding masks for target
            tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
            tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
            tgt_mask = tgt_causal_mask & tgt_padding_mask
            
            # Forward pass with no_grad for better performance
            with torch.no_grad():
                output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
            
            # Get next token
            next_token_logits = output[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(1)
            
            # Stop if EOS token generated
            if next_token.item() == self.tgt_vocab.token2idx['<EOS>']:
                break
            
            # Append to sequence
            tgt_tokens = torch.cat([tgt_tokens, next_token], dim=1)
        
        return self.tgt_vocab.decode(tgt_tokens[0].tolist())
    
    def beam_search(self, src_sentence, beam_width=3, max_len=50):
        """Beam search decoding - maintain multiple hypotheses"""
        # Encode source with consistent max_len
        src_tokens = torch.tensor([self.src_vocab.encode(src_sentence, max_len=15)], dtype=torch.long)
        src_mask = self.model.create_padding_mask(src_tokens)
        
        # Initialize beam with SOS token
        beams = [{'tokens': [self.tgt_vocab.token2idx['<SOS>']], 'score': 0.0}]
        
        for step in range(max_len):
            candidates = []
            
            for beam in beams:
                if beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>']:
                    candidates.append(beam)
                    continue
                
                # Get current sequence
                tgt_tokens = torch.tensor([beam['tokens']], dtype=torch.long)
                # Create both causal and padding masks for target
                tgt_causal_mask = self.model.create_causal_mask(tgt_tokens.size(1))
                tgt_padding_mask = self.model.create_padding_mask(tgt_tokens)
                tgt_mask = tgt_causal_mask & tgt_padding_mask
                
                # Forward pass with no_grad for better performance
                with torch.no_grad():
                    output = self.model(src_tokens, tgt_tokens, src_mask, tgt_mask)
                
                # Get probabilities for next token
                logits = output[:, -1, :]
                
                # Calculate log probabilities for the next token
                probs = F.log_softmax(logits, dim=-1)
                
                # Get the top 'beam_width' candidates (probabilities and indices)
                top_probs, top_indices = torch.topk(probs, beam_width)
                
                # Create new candidate beams by extending the current beam
                for prob, idx in zip(top_probs[0], top_indices[0]):
                    new_tokens = beam['tokens'] + [idx.item()]
                    new_score = beam['score'] + prob.item()
                    candidates.append({'tokens': new_tokens, 'score': new_score})
            
            # Select top beam_width candidates
            candidates.sort(key=lambda x: x['score'], reverse=True)
            beams = candidates[:beam_width]
            
            # Check if all beams ended
            if all(beam['tokens'][-1] == self.tgt_vocab.token2idx['<EOS>'] for beam in beams):
                break
        
        # Return best sequence
        best_beam = max(beams, key=lambda x: x['score'])
        return self.tgt_vocab.decode(best_beam['tokens'])

## main.py

import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Greedy and beam search
        start_time = time.time()
        greedy_result = inference.greedy_decode(sentence)
        greedy_time = time.time() - start_time
        
        start_time = time.time()
        beam_result = inference.beam_search(sentence, beam_width=3)
        beam_time = time.time() - start_time
        
        print(f"Greedy: '{greedy_result}' (time: {greedy_time:.3f}s)")
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()


## Exploring Beam Search Tradeoffs

Well done! We're finishing our journey through transformer inference strategies by exploring how different beam widths affect both output quality and inference speed. In this exercise, you'll modify the inference section to systematically test various beam widths and observe their impact.

---

### Your Task

Your job is to update the provided code to test multiple beam widths instead of just using a fixed value of 3. You'll need to:

- Create a list of different beam widths to test (such as 1, 3, and 5)
- Replace the single beam search call with a loop that tries each beam width
- For each beam width, measure the execution time and print the results along with the beam width used

This hands-on comparison will give you valuable insights into the trade-offs between generation quality and computational cost!

---

### Starter Code

```python
import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    # TODO: Create a list of beam widths to test (e.g., [1, 3, 5])
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # TODO: Tests different beam widths
        
        print(f"Beam: '{beam_result}' (time: {beam_time:.3f}s)")
        print("-" * 50)
    

if __name__ == "__main__":
    main()
```


---

### Solution

The inference section called `beam_search` once per sentence with a fixed width, and the `print` line used `beam_result` and `beam_time` that were never defined. The fix creates a list of beam widths and runs the timed `beam_search` call inside a loop over it:

```python
# Create a list of beam widths to test
beam_widths = [1, 3, 5]

# Test different beam widths
for beam_width in beam_widths:
    start_time = time.time()
    beam_result = inference.beam_search(sentence, beam_width=beam_width)
    beam_time = time.time() - start_time

    print(f"Beam (width={beam_width}): '{beam_result}' (time: {beam_time:.3f}s)")
```

- `beam_widths = [1, 3, 5]` is defined once, before the sentence loop, so the same widths are tested for every sentence.
- The inner `for beam_width in beam_widths` loop runs `beam_search` with each width. A width of `1` keeps a single hypothesis, so it behaves like greedy decoding, while larger widths keep more candidates.
- `time.time()` is read before and after each call, so `beam_time` measures only that beam search run.
- The `print` line moved inside the inner loop and now includes `beam_width`, so each result is labelled with the width that produced it.
- `beam_result` and `beam_time` are assigned inside the loop. Without the loop the `print` line raises a `NameError`.

When the script runs, it trains the small model for 2 epochs. For each of the three test sentences it then prints one line per beam width, together with the time each one took.

The notebook version needs the helper modules (`transformer.model`, `data_utils`, `train`, `inference`) to be available, as in the CodeSignal project.


In [ ]:
import torch
from torch.utils.data import DataLoader
import time

from transformer.model import Transformer
from data_utils import create_synthetic_data, Vocabulary, TranslationDataset
from train import TransformerTrainer
from inference import TransformerInference

def collate_fn(batch):
    """Custom collate function"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]
    tgt_output_batch = [item['tgt_output'] for item in batch]
    
    src_batch = torch.nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_batch = torch.nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    tgt_output_batch = torch.nn.utils.rnn.pad_sequence(tgt_output_batch, batch_first=True, padding_value=0)
    
    return {'src': src_batch, 'tgt': tgt_batch, 'tgt_output': tgt_output_batch}

def main():
    """Test inference with trained model"""
    print("Testing Transformer Inference...")
    
    # Prepare data and train model
    src_sentences, tgt_sentences = create_synthetic_data(num_samples=200)
    
    src_vocab = Vocabulary()
    tgt_vocab = Vocabulary()
    src_vocab.build_vocab(src_sentences)
    tgt_vocab.build_vocab(tgt_sentences)
    
    # Create and train model
    dataset = TranslationDataset(src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=15)
    train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    
    model = Transformer(
        src_vocab_size=src_vocab.size,
        tgt_vocab_size=tgt_vocab.size,
        d_model=64,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=256,
        dropout=0.1
    )
    
    # Quick training
    trainer = TransformerTrainer(model, train_loader, lr=1e-3, warmup_steps=25)
    print("Quick training for 2 epochs...")
    for epoch in range(2):
        avg_loss = trainer.train_epoch()
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")
    
    # Test inference
    inference = TransformerInference(model, src_vocab, tgt_vocab)
    
    test_sentences = ["hello world", "good morning", "thank you very much"]
    
    print("\nInference Results:")
    # Create a list of beam widths to test
    beam_widths = [1, 3, 5]
    for sentence in test_sentences:
        print(f"Source: '{sentence}'")
        print(f"Expected: '{' '.join(sentence.split()[::-1])}'")
        
        # Test different beam widths
        for beam_width in beam_widths:
            start_time = time.time()
            beam_result = inference.beam_search(sentence, beam_width=beam_width)
            beam_time = time.time() - start_time
            
            print(f"Beam (width={beam_width}): '{beam_result}' (time: {beam_time:.3f}s)")
        
        print("-" * 50)
    

if __name__ == "__main__":
    main()
